# 19_observability: executable lesson notebook

Read [the guide](README.md). These cells mirror every script and run offline by default.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_agent_metrics

Source: `modules/19_observability/examples/01_agent_metrics.py`.

In [ ]:
# Source: modules/19_observability/examples/01_agent_metrics.py
"""Inspect tokens, cycles, latency and actual tool calls: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/01_agent_metrics.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.lessons import complete, lesson_settings
from common.pretty import estimate_cost, run_cli
from common.production_support import answer_support


def main() -> None:
    _, cfg = lesson_settings()
    result = asyncio.run(answer_support({"prompt": "Refund processing time?"}, cfg))
    # KEY IDEA: inspect measured tool success and cycles, not just the final prose.
    print({k: v for k, v in result.items() if k != "answer"})
    print("Inference estimate USD:", estimate_cost(result["usage"], cfg))
    complete()


run_lesson(main, "modules/19_observability/examples/01_agent_metrics.py")

## 02_local_tracing

Source: `modules/19_observability/examples/02_local_tracing.py`.

In [ ]:
# Source: modules/19_observability/examples/02_local_tracing.py
"""Console and real local OTLP spans: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/02_local_tracing.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import argparse
import os
import subprocess
import sys

from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--otlp", help="Local Jaeger endpoint, http://127.0.0.1:4318/v1/traces")
    args, cfg = lesson_settings(parser)
    # KEY IDEA: process globals require a fresh tracer process, including in notebooks.
    command = [sys.executable, "-X", "utf8", "-m", "common.telemetry_lab"]
    if args.mock:
        command.append("--mock")
    if args.otlp:
        command.extend(["--otlp", args.otlp])
    subprocess.run(command, env={**os.environ, "PYTHONUTF8": "1"}, check=True, timeout=60)
    complete()


run_lesson(main, "modules/19_observability/examples/02_local_tracing.py")

## 03_offline_testing

Source: `modules/19_observability/examples/03_offline_testing.py`.

In [ ]:
# Source: modules/19_observability/examples/03_offline_testing.py
"""Tool unit checks and moto session storage: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/03_offline_testing.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.production_checks import offline_checks


def main() -> None:
    lesson_settings()
    # KEY IDEA: a tool test isolates business logic before any model is involved.
    print(offline_checks())
    complete()


run_lesson(main, "modules/19_observability/examples/03_offline_testing.py")

## 04_strands_evals

Source: `modules/19_observability/examples/04_strands_evals.py`.

In [ ]:
# Source: modules/19_observability/examples/04_strands_evals.py
"""Deterministic cases and separate LLM judges: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/04_strands_evals.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.support_evaluation import evaluate_support, judge_demo


def main() -> None:
    _, cfg = lesson_settings()
    report = asyncio.run(evaluate_support(cfg))
    print("Deterministic overall score:", report.overall_score)
    # KEY IDEA: scripted judge scores test interfaces; live judges need calibration.
    print("Output and trajectory judges:", asyncio.run(judge_demo(cfg)))
    complete()


run_lesson(main, "modules/19_observability/examples/04_strands_evals.py")

## 05_evaluation_cli

Source: `modules/19_observability/examples/05_evaluation_cli.py`.

In [ ]:
# Source: modules/19_observability/examples/05_evaluation_cli.py
"""Actual Evals CLI exit codes: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/05_evaluation_cli.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.evaluation_cli import gate
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    baseline = gate(live=cfg.provider != "mock")
    print("Baseline:", baseline)
    if baseline["exit_code"]:
        raise SystemExit(baseline["exit_code"])
    # KEY IDEA: deliberately break an answer and check that the real CLI exits nonzero.
    regression = gate(regression=True)
    print("Injected regression:", regression)
    assert regression["exit_code"] == 1
    complete()


run_lesson(main, "modules/19_observability/examples/05_evaluation_cli.py")

## 06_cloud_observability

Source: `modules/19_observability/examples/06_cloud_observability.py`.

In [ ]:
# Source: modules/19_observability/examples/06_cloud_observability.py
"""Content-free CloudWatch metrics: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/06_cloud_observability.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.lessons import complete
from common.pretty import run_cli
from common.production_checks import emit_metrics
from common.production_lessons import aws_settings
from common.production_support import answer_support


def main() -> None:
    _, cfg = aws_settings()
    result = asyncio.run(answer_support({"prompt": "Refund time?"}, cfg))
    # KEY IDEA: CloudWatch receives numeric usage/latency, never the prompt or answer.
    print(emit_metrics(cfg, result))
    complete()


run_lesson(main, "modules/19_observability/examples/06_cloud_observability.py")

## 07_managed_evaluation

Source: `modules/19_observability/examples/07_managed_evaluation.py`.

In [ ]:
# Source: modules/19_observability/examples/07_managed_evaluation.py
"""Managed on-demand evaluation: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/examples/07_managed_evaluation.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import argparse
from pathlib import Path

from common.lessons import complete
from common.managed_evaluation import evaluate_spans
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--spans", type=Path, help="Reviewed CloudWatch export for one synthetic session"
    )
    args, cfg = aws_settings(parser)
    # KEY IDEA: metadata-only traces cannot establish answer helpfulness.
    print(evaluate_spans(cfg, args.spans))
    complete()


run_lesson(main, "modules/19_observability/examples/07_managed_evaluation.py")

## support_quality_gate

Source: `modules/19_observability/use_case/support_quality_gate.py`.

In [ ]:
# Source: modules/19_observability/use_case/support_quality_gate.py
"""Support CI quality gate: Nimbus production lesson.

Run: uv run --all-groups python modules/19_observability/use_case/support_quality_gate.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import argparse

from common.evaluation_cli import gate
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--regression", action="store_true")
    args, cfg = lesson_settings(parser)
    # KEY IDEA: return the CLI failure code to the CI runner without swallowing it.
    result = gate(regression=args.regression, live=cfg.provider != "mock")
    print(result)
    if result["exit_code"]:
        raise SystemExit(result["exit_code"])
    complete()


run_lesson(main, "modules/19_observability/use_case/support_quality_gate.py")